In [1]:
# ================================
# 1️⃣ Install Dependencies
# ================================
!pip install transformers datasets scikit-learn tqdm

# ================================
# 2️⃣ Imports
# ================================
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from tqdm import tqdm

# For auto-download (Colab)
from google.colab import files

# ================================
# 3️⃣ Load Dataset
# ================================
file_path = "/content/ML_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

print("Total samples:", len(data))

# ================================
# 4️⃣ Split: 3-shot + rest
# ================================
few_shot_data = data[:3]     # ONLY first 3 samples
test_data = data[3:]         # Rest for prediction

print("Few-shot samples:", len(few_shot_data))
print("Test samples:", len(test_data))

# ================================
# 5️⃣ Load SciBERT
# ================================
model_name = "allenai/scibert_scivocab_cased"  # 👈 recommended

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# ================================
# 6️⃣ Get Sentence Embedding
# ================================
def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # CLS token embedding
    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    return embedding

# ================================
# 7️⃣ Encode Few-Shot Examples
# ================================
few_embeddings = []
few_labels = []

for ex in few_shot_data:
    emb = get_embedding(ex["input"])
    few_embeddings.append(emb)
    few_labels.append(ex["output"].strip())

few_embeddings = np.vstack(few_embeddings)

# ================================
# 8️⃣ Prediction Function (Similarity)
# ================================
def predict(text):
    emb = get_embedding(text)

    sims = cosine_similarity(emb, few_embeddings)[0]
    best_idx = np.argmax(sims)

    return few_labels[best_idx]

# ================================
# 9️⃣ Run Predictions
# ================================
y_true = []
y_pred = []
results = []

for ex in tqdm(test_data):
    text = ex["input"]
    gt = ex["output"].strip()

    pred = predict(text)

    y_true.append(gt)
    y_pred.append(pred)

    results.append({
        "input": text,
        "ground_truth": gt,
        "predicted": pred
    })

# ================================
# 🔟 Save Results
# ================================
output_file = "/content/scibert_3shot_results.json"

with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

files.download(output_file)

print(f"\n✅ Results saved to: {output_file}")

# ================================
# 1️⃣1️⃣ Metrics
# ================================
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== SCIBERT 3-SHOT RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_true, y_pred))

# ================================
# 1️⃣2️⃣ Show Sample Predictions
# ================================
for i in range(min(5, len(results))):
    print("\n--- Example ---")
    print("Input :", results[i]["input"])
    print("GT    :", results[i]["ground_truth"])
    print("Pred  :", results[i]["predicted"])

Total samples: 1962
Few-shot samples: 3
Test samples: 1959


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/442M [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: allenai/scibert_scivocab_cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 1959/1959 [00:19<00:00, 99.27it/s] 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Results saved to: /content/scibert_3shot_results.json

===== SCIBERT 3-SHOT RESULTS =====
Accuracy : 0.1440
Precision: 0.0207
Recall   : 0.1440
F1 Score : 0.0362

===== Classification Report =====
              precision    recall  f1-score   support

    based_on       0.00      0.00      0.00        90
  implements       0.00      0.00      0.00       170
    improves       0.00      0.00      0.00       242
 no_relation       0.00      0.00      0.00       584
     part_of       0.14      1.00      0.25       282
    used_for       0.00      0.00      0.00       591

    accuracy                           0.14      1959
   macro avg       0.02      0.17      0.04      1959
weighted avg       0.02      0.14      0.04      1959


===== Confusion Matrix =====
[[  0   0   0   0  90   0]
 [  0   0   0   0 170   0]
 [  0   0   0   0 242   0]
 [  0   0   0   0 584   0]
 [  0   0   0   0 282   0]
 [  0   0   0   0 591   0]]

--- Example ---
Input : Sentence: Deep learning has enhanced po

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
